# 18b Fine-tune the student instead of freezing it (Google Colab, GPU)

Notebook 18 trained a logistic regression on **frozen** embeddings from labels written by a language model, and found the student 9.4 points behind gold-label training with a smaller labeling model (Claude Haiku) and 4.9 points behind with a larger one (Claude Opus). This notebook asks whether that gap is a property of the frozen student. It **fine-tunes all the weights** of the same encoder (`BAAI/bge-small-en-v1.5`, 33M parameters) on the same 2,000 Banking77 messages, once per label source (human, Haiku, Opus), three seeds each, and scores on the same 3,080 test messages.

**Runtime.** Colab: Runtime, Change runtime type, T4 GPU. About 10 minutes for the full run (18 fine-tuning runs of 12 epochs). On CPU it works but takes hours, which is why it lives here.

**Fixed in advance (no tuning on the test set).** Learning rate 1e-4, batch size 32, 12 epochs, linear decay with 10% warmup, maximum length 64 tokens, mixed precision. These are common defaults for a small encoder, chosen before the first run and not changed after seeing a result. If you change one, say so next to the numbers.

**Files you need.** Three small files from the companion repo folder `data/results/18_distillation/`: `sample_idx.json` (which 2,000 training messages were labeled), `teacher_labels.jsonl` (Haiku) and `teacher_labels_opus.jsonl` (Opus). The notebook tries to download them from GitHub and otherwise asks you to upload them.

In [ ]:
SMOKE = False   # True: quick check that the code runs (300 messages, 4 epochs, 300 test messages, one seed). Leave False for the real run.
import os
SMOKE = SMOKE or os.environ.get("SMOKE") == "1"
if not SMOKE:
    !pip -q install transformers==4.57.6 2>&1 | tail -n 1
import json, math, time, csv, urllib.request, collections
from pathlib import Path
import numpy as np, torch
from scipy.stats import binomtest
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU, slow)")
import transformers, sys
_gpu = torch.cuda.is_available()
print("preflight | python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__,
      "| GPU:", (torch.cuda.get_device_name(0) + f", {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB") if _gpu else "none", flush=True)
if not SMOKE and not _gpu:
    raise SystemExit("No GPU. In Colab: Runtime, Change runtime type, choose a GPU (T4 is enough), then run all again. CPU and TPU runtimes are not supported.")

**Goal.** Download Banking77 (77 intents, 10,003 training and 3,080 test messages; Casanueva et al. 2020, repository labeled CC-BY-4.0) and the three label files. The training messages that were labeled by the language models are a random 2,000 of the 10,003; `sample_idx.json` lists them, so human, Haiku and Opus labels all refer to the same messages.

In [ ]:
DATA = Path("banking77"); DATA.mkdir(exist_ok=True)
BASE = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/"
for s in ("train", "test"):
    if not (DATA / f"{s}.csv").exists(): urllib.request.urlretrieve(BASE + f"{s}.csv", DATA / f"{s}.csv")
tr = list(csv.DictReader(open(DATA / "train.csv"))); te = list(csv.DictReader(open(DATA / "test.csv")))
classes = sorted({r["category"] for r in tr}); cid = {c: i for i, c in enumerate(classes)}; K = len(classes)
ytr = np.array([cid[r["category"]] for r in tr]); yte = np.array([cid[r["category"]] for r in te])
print(len(tr), "training,", len(te), "test,", K, "intents")

REPO = "https://raw.githubusercontent.com/benman1/deterministic-ai-workshop/main/data/results/18_distillation/"
need = ["sample_idx.json", "teacher_labels.jsonl", "teacher_labels_opus.jsonl"]
LAB = Path("labels"); LAB.mkdir(exist_ok=True)
missing = []
for f in need:
    if not (LAB / f).exists():
        try: urllib.request.urlretrieve(REPO + f, LAB / f)
        except Exception: missing.append(f)
if missing:
    print("Not on GitHub yet:", missing, "- upload them from data/results/18_distillation/ in the companion repo.")
    from google.colab import files
    for name, data in files.upload().items(): (LAB / name).write_bytes(data)
idx = np.array(json.load(open(LAB / "sample_idx.json")))
def labels(fname):
    T = {r["id"]: r["label"] for r in map(json.loads, open(LAB / fname))}
    return np.array([cid.get(T[i], -1) for i in range(len(idx))])
gold = ytr[idx]; haiku = labels("teacher_labels.jsonl"); opus = labels("teacher_labels_opus.jsonl")
for name, l in (("Haiku", haiku), ("Opus", opus)):
    print(f"{name}: right on {int((l == gold).sum())} of {len(idx)}; labels outside the 77 names: {int((l < 0).sum())}")
assert (haiku >= 0).sum() >= 1990 and (opus >= 0).sum() >= 1990

**Goal.** Define the fine-tuning run and the metrics. **Wilson interval**: the interval for a proportion that stays inside 0 to 1 and is honest when the count is small. **Exact McNemar test**: compares two models on the same test messages by counting only the messages one gets right and the other wrong. **ECE** (expected calibration error, 10 equal-width bins): the average gap between a model's confidence and how often it is right.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL = "BAAI/bge-small-en-v1.5"
dev = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL)
LR, BS, EPOCHS, MAXLEN = 1e-4, 32, (4 if SMOKE else 12), 64
test_n = 300 if SMOKE else len(te)
def enc(texts): return tok(texts, padding=True, truncation=True, max_length=MAXLEN, return_tensors="pt")
sub = np.random.default_rng(123).permutation(len(te))[:test_n]   # Banking77 test is sorted by intent: never take the first rows
test_texts = [te[i]["text"] for i in sub]; y_test = yte[sub]

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h
def ece(P, y, bins=10):
    conf = P.max(1); ok = P.argmax(1) == y; tot = 0
    for b in range(bins):
        i = (conf >= b / bins) & ((conf < (b + 1) / bins) | (b == bins - 1))
        if i.any(): tot += i.sum() * abs(conf[i].mean() - ok[i].mean())
    return tot / len(y)

def finetune(texts, y, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    m = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=K).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    steps = EPOCHS * math.ceil(len(texts) / BS)
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))
    keep = np.where(y >= 0)[0]   # drop the rare label outside the 77 names
    for ep in range(EPOCHS):
        m.train(); perm = np.random.permutation(keep)
        for b in range(0, len(perm), BS):
            ii = perm[b:b + BS]; x = enc([texts[i] for i in ii]).to(dev)
            with torch.autocast(dev, dtype=torch.float16, enabled=(dev == "cuda")):
                loss = torch.nn.functional.cross_entropy(m(**x).logits.float(), torch.tensor(y[ii]).to(dev))
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
    m.eval(); P = []
    with torch.no_grad():
        for b in range(0, test_n, 128):
            x = enc(test_texts[b:b + 128]).to(dev)
            P.append(torch.softmax(m(**x).logits.float(), -1).cpu().numpy())
    del m; torch.cuda.empty_cache() if dev == "cuda" else None
    return np.concatenate(P)

**Goal.** Run every combination of training size, label source and seed, and print one line per run as it finishes. A run that prints nothing for several minutes means something is wrong: stop it and check the GPU line above. The full run is 2 sizes x 3 label sources x 3 seeds = 18 fine-tunings.

In [ ]:
train_texts_all = [tr[i]["text"] for i in idx]
sizes = (300,) if SMOKE else (500, 2000)
seeds = (0,) if SMOKE else (0, 1, 2)
sources = {"human": gold, "Haiku": haiku, "Opus": opus}
runs = {}   # (size, source, seed) -> test probabilities
t0 = time.time()
for N in sizes:
    for seed in seeds:
        sel = np.random.default_rng(seed).permutation(len(idx))[:N]   # same subset for all three sources
        for name, lab in sources.items():
            P = finetune([train_texts_all[i] for i in sel], lab[sel], seed)
            runs[(N, name, seed)] = P
            ok = P.argmax(1) == y_test
            print(f"N={N:5d} {name:6s} seed {seed}: accuracy {ok.mean():.3f}  ECE {ece(P, y_test):.3f}   [{time.time() - t0:5.0f} s]", flush=True)

**Goal.** Summarize. Accuracy is the mean over seeds; the Wilson interval is for the pooled count of correct answers over seeds (3 x 3,080 test messages, which overstates how much the three seeds add, because the test messages repeat: read it as a rough band, not a guarantee). The exact McNemar test compares two label sources on the same test messages for each seed separately. The last column repeats the frozen-embedding result from notebook 18.

In [ ]:
FROZEN = {"human": 0.895, "Haiku": 0.801, "Opus": 0.846}   # notebook 18, logistic regression on frozen embeddings, N=2000
print(f"{'N':>5s} {'labels':>7s} {'accuracy (mean of seeds)':>26s} {'Wilson':>16s} {'ECE':>6s}  {'frozen (nb 18)':>14s}")
for N in sizes:
    for name in sources:
        accs = [(runs[(N, name, s)].argmax(1) == y_test).mean() for s in seeds]
        k = sum(int((runs[(N, name, s)].argmax(1) == y_test).sum()) for s in seeds); n = len(seeds) * test_n; lo, hi = wilson(k, n)
        e = np.mean([ece(runs[(N, name, s)], y_test) for s in seeds])
        print(f"{N:5d} {name:>7s} {np.mean(accs):14.3f} (+/- {np.std(accs):.3f}) {lo:7.3f} to {hi:5.3f} {e:6.3f}  {FROZEN[name] if N == 2000 else float('nan'):14.3f}")
print()
for N in sizes:
    for a, b in (("human", "Haiku"), ("Opus", "Haiku"), ("human", "Opus")):
        for s in seeds:
            oa = runs[(N, a, s)].argmax(1) == y_test; ob = runs[(N, b, s)].argmax(1) == y_test
            n10, n01 = int((oa & ~ob).sum()), int((~oa & ob).sum())
            p = binomtest(n10, n10 + n01, 0.5).pvalue if n10 + n01 else 1.0
            print(f"N={N:5d} seed {s}: {a:5s} vs {b:5s}: only {a} right {n10:4d}, only {b} right {n01:4d}, exact McNemar p = {p:.2g}")
json.dump({f"{N}|{n}|{s}": dict(acc=float((P.argmax(1) == y_test).mean()), ece=float(ece(P, y_test))) for (N, n, s), P in runs.items()},
          open("18b_finetune_results.json", "w"), indent=1)

**What to read from the table.** Compare, at 2,000 messages, the gap between the human-label and the Haiku-label rows here with the 9.4-point gap in notebook 18 (frozen embeddings), and the gap between Opus and human labels with 4.9 points. Three outcomes are possible and each means something different for the book: the gaps shrink (a fine-tuned student can learn to ignore some of the teacher's mistakes), stay about the same (the label noise, not the student, sets the limit), or grow (a student with more capacity memorizes the teacher's mistakes). Write down which happened before reading anything else into the numbers.

**Limits.** One encoder, fixed hyperparameters, three seeds, and the three label sets are single labeling passes. The ECE column compares models trained with the same recipe; do not compare it with notebook 18's, whose student is a different kind of model. Colab's GPU type and library versions change the last digit of an accuracy, not its meaning; the seed controls the batch order and the classifier's starting weights.

The file `18b_finetune_results.json` is written next to the notebook. Download it (left sidebar, Files) and put it in `data/results/` of the companion repo so the slides and the book can quote it.

**Goal.** Save everything. The zip holds the result table, the predicted probabilities of every run (so the accuracy, ECE and McNemar numbers can be recomputed, and other metrics added, without retraining), the test labels and subset, and `run_manifest.json` with the settings, the software and GPU, the encoder's revision, and hashes of the training texts and the three label sets. Colab offers it as a download; download it before closing the session.

In [ ]:
import sys, platform, hashlib, zipfile, shutil, datetime, importlib.metadata as md
COPY_TO_DRIVE = False   # True: also copy the zip to Google Drive (Colab asks you to sign in)

def package_version(name):
    try: return md.version(name)
    except Exception: return None

def hub_revision(kind, repo):
    """The commit of a Hugging Face model or dataset that was downloaded, so the run can be reproduced."""
    try:
        from huggingface_hub import HfApi
        api = HfApi(); return (api.dataset_info(repo) if kind == "dataset" else api.model_info(repo)).sha
    except Exception as e:
        return "unavailable: " + repr(e)[:60]

def sha256_of(obj): return hashlib.sha256(json.dumps(obj, sort_keys=True, default=str).encode()).hexdigest()

def environment():
    gpu = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
    return dict(finished=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), python=sys.version.split()[0], platform=platform.platform(),
                torch=torch.__version__, cuda=torch.version.cuda, gpu=gpu.name if gpu else None, gpu_gb=round(gpu.total_memory / 1e9, 1) if gpu else None,
                packages={p: package_version(p) for p in ("transformers", "peft", "accelerate", "bitsandbytes", "huggingface_hub", "scipy", "numpy", "pandas")}, smoke=SMOKE)

def save_bundle(zip_name, paths, manifest):
    """Write run_manifest.json, zip it with the listed files and folders, print what is inside, offer the download."""
    json.dump(manifest, open("run_manifest.json", "w"), indent=1, default=str)
    with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
        for p in ["run_manifest.json"] + paths:
            if os.path.isdir(p):
                for root, _, names in os.walk(p):
                    for n in names: z.write(os.path.join(root, n))
            elif os.path.exists(p): z.write(p)
            else: print("missing, not zipped:", p)
    with zipfile.ZipFile(zip_name) as z:
        print(f"wrote {zip_name} ({os.path.getsize(zip_name) / 1e6:.1f} MB). Contents:")
        for i in z.infolist(): print(f"  {i.file_size / 1e6:8.2f} MB  {i.filename}")
    if COPY_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive"); shutil.copy(zip_name, "/content/drive/MyDrive/"); print("copied to My Drive")
    try:
        from google.colab import files as colab_files; colab_files.download(zip_name)
    except Exception:
        print("Not in Colab, or the browser blocked the download: take the zip from the Files panel.")

np.savez_compressed("18b_probabilities.npz", y_test=y_test, test_subset=np.array(sub), train_idx=np.array(idx), gold=np.array(gold), haiku=np.array(haiku), opus=np.array(opus),
                    **{f"p_{N}_{n}_{s}": P.astype(np.float32) for (N, n, s), P in runs.items()})
manifest = dict(environment=environment(), model=MODEL, model_revision=hub_revision("model", MODEL),
                settings=dict(lr=LR, batch_size=BS, epochs=EPOCHS, max_len=MAXLEN, sizes=list(sizes), seeds=list(seeds), test_n=test_n, labels=K),
                train_texts_sha256=sha256_of(train_texts_all), labels_sha256={k: sha256_of(np.asarray(v).tolist()) for k, v in sources.items()}, runs=len(runs))
save_bundle("18b_finetune_bundle.zip", ["18b_finetune_results.json", "18b_probabilities.npz"], manifest)